# Dias meteorológicos representativos — 2019

Este notebook seleciona dois dias representativos por estação por meio de K-means e produz o JSON consumido pelo modelo OPF. Somente a versão corrigida do notebook original foi preservada.

## Fluxo

1. preparar o ambiente e localizar os 12 perfis mensais;
2. validar o calendário e a grade de 10 minutos;
3. construir um vetor diário com irradiância e temperatura;
4. executar K-means separadamente em cada estação;
5. escolher o dia observado mais próximo de cada centroide;
6. exportar tabelas, figuras, manifesto e o JSON com oito dias.

## 1. Configuração

In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import re
import subprocess
import sys
import unicodedata
from pathlib import Path
from types import SimpleNamespace

IN_COLAB = "google.colab" in sys.modules
CFG = SimpleNamespace(
    repository_url="https://github.com/DarleneJD/ACOPF.git",
    repository_dir=(Path("/content/ACOPF") if IN_COLAB else Path.cwd()),
    output_dir=(
        Path("/content/drive/MyDrive/ACOPF/dias_representativos_2019")
        if IN_COLAB else Path.cwd()/"outputs"/"dias_representativos_2019"
    ),
    data_year=2019,
    clusters_per_season=2,
    random_seed=42,
    kmeans_initializations=50,
    minimum_daily_coverage=0.90,
)
CFG

## 2. Ambiente e dados

In [ ]:
required = {
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing],
        check=True,
    )

if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive", force_remount=False)

if not (CFG.repository_dir/"data"/"meteorology"/"profiles_10min").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", CFG.repository_url,
         str(CFG.repository_dir)],
        check=True,
    )

profiles_dir = CFG.repository_dir/"data"/"meteorology"/"profiles_10min"
expected_profiles = [
    profiles_dir/f"perfis_10min_{CFG.data_year}_{month:02d}.csv"
    for month in range(1, 13)
]
missing_profiles = [path for path in expected_profiles if not path.is_file()]
if missing_profiles:
    formatted = "\n".join(f"  - {path}" for path in missing_profiles)
    raise FileNotFoundError(
        "A estrutura publicada do repositório está incompleta. "
        f"Perfis mensais ausentes:\n{formatted}")

CFG.output_dir.mkdir(parents=True, exist_ok=True)
print("Repositório:", CFG.repository_dir)
print("Saídas:", CFG.output_dir)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

PASTA_CSV = CFG.repository_dir/"data"/"meteorology"/"profiles_10min"
PASTA_SAIDA = CFG.output_dir
ANO_DADOS = CFG.data_year
ANO_METEOROLOGICO = ANO_DADOS
N_CLUSTERS_ESTACAO = CFG.clusters_per_season
SEED = CFG.random_seed
N_INIT = CFG.kmeans_initializations
COBERTURA_MINIMA = CFG.minimum_daily_coverage
PONTOS_POR_DIA = 144

ESTACOES = {
    "Verao": (12, 1, 2),
    "Outono": (3, 4, 5),
    "Inverno": (6, 7, 8),
    "Primavera": (9, 10, 11),
}

COLUNAS_OBRIGATORIAS = {
    "timestamp", "source_year_index", "month", "day", "hour", "minute",
    "irradiance_pu", "ambient_temperature_c",
}

## 3. Leitura e validação

Os arquivos devem se chamar `perfis_10min_2019_MM.csv`. O ano 2019 é validado no nome, no timestamp e na coluna `source_year_index`.

In [ ]:
def nome_simples(texto: object) -> str:
    """Normaliza nomes de colunas sem alterar os dados."""
    texto = (
        unicodedata.normalize("NFKD", str(texto))
        .encode("ascii", "ignore")
        .decode()
    )
    return re.sub(r"[^a-z0-9]+", "_", texto.lower()).strip("_")

def mes_do_arquivo(caminho: Path) -> int | None:
    achado = re.search(
        rf"_{ANO_DADOS}_(0[1-9]|1[0-2])\.csv$", caminho.name, flags=re.I
    )
    return int(achado.group(1)) if achado else None

def _colunas_normalizadas(df: pd.DataFrame, arquivo: Path) -> dict[str, str]:
    mapa = {nome_simples(c): c for c in df.columns}
    ausentes = sorted(COLUNAS_OBRIGATORIAS.difference(mapa))
    if ausentes:
        raise ValueError(f"{arquivo.name}: colunas obrigatorias ausentes: {ausentes}")
    return mapa

def ler_arquivos(pasta: Path = PASTA_CSV) -> pd.DataFrame:
    """Lê e valida os doze perfis mensais de 2019."""
    partes = []
    arquivos = sorted(pasta.glob(f"perfis_10min_{ANO_DADOS}_*.csv"))
    for arquivo in arquivos:
        mes_arquivo = mes_do_arquivo(arquivo)
        if mes_arquivo is None:
            continue
        df = pd.read_csv(arquivo)
        col = _colunas_normalizadas(df, arquivo)
        timestamp = pd.to_datetime(
            df[col["timestamp"]], format="%Y-%m-%dT%H:%M:%S", errors="coerce"
        )
        if timestamp.isna().any():
            raise ValueError(f"{arquivo.name}: timestamp inválido")
        numericas = {
            nome: pd.to_numeric(df[col[nome]], errors="coerce")
            for nome in (
                "source_year_index", "month", "day", "hour", "minute",
                "irradiance_pu", "ambient_temperature_c",
            )
        }
        if any(serie.isna().any() for serie in numericas.values()):
            raise ValueError(f"{arquivo.name}: valores numéricos inválidos")
        validacoes = {
            "ano": timestamp.dt.year.eq(ANO_DADOS),
            "source_year_index": numericas["source_year_index"].eq(ANO_DADOS),
            "month": numericas["month"].eq(timestamp.dt.month),
            "day": numericas["day"].eq(timestamp.dt.day),
            "hour": numericas["hour"].eq(timestamp.dt.hour),
            "minute": numericas["minute"].eq(timestamp.dt.minute),
            "mes_do_arquivo": timestamp.dt.month.eq(mes_arquivo),
        }
        inconsistencias = [nome for nome, ok in validacoes.items() if not ok.all()]
        if inconsistencias:
            raise ValueError(f"{arquivo.name}: inconsistências: {inconsistencias}")
        partes.append(pd.DataFrame({
            "timestamp": timestamp,
            "irradiance_pu": numericas["irradiance_pu"].astype(float),
            "ambient_temperature_c": numericas["ambient_temperature_c"].astype(float),
            "arquivo_origem": arquivo.name,
            "mes_arquivo": mes_arquivo,
        }))
        print(f"Lido {arquivo.name}: {len(df):,} amostras")
    meses = sorted({int(p["mes_arquivo"].iloc[0]) for p in partes})
    if meses != list(range(1, 13)):
        raise FileNotFoundError(f"Esperados 12 meses; encontrados: {meses}")
    return pd.concat(partes, ignore_index=True).sort_values("timestamp").reset_index(drop=True)

## 4. Representação de cada dia

Cada dia válido é reamostrado em 144 pontos. O vetor usado no agrupamento contém 144 valores de irradiância seguidos por 144 valores de temperatura ambiente.

In [ ]:
def grade_normalizada_do_dia(dados_dia: pd.DataFrame) -> pd.DataFrame | None:
    """Produz a grade de 144 pontos usada tanto no K-means quanto no JSON."""
    if dados_dia.empty:
        return None

    data = dados_dia["timestamp"].dt.normalize().iloc[0]
    grade = pd.date_range(data, periods=PONTOS_POR_DIA, freq="10min")
    dia = (
        dados_dia.groupby("timestamp")[
            ["irradiance_pu", "ambient_temperature_c"]
        ]
        .mean(numeric_only=True)
        .reindex(grade)
    )
    cobertura = float(dia.notna().all(axis=1).mean())
    if cobertura < COBERTURA_MINIMA:
        return None

    dia = dia.interpolate(method="time", limit_direction="both")
    if dia.isna().any().any():
        return None
    dia.index.name = "timestamp"
    return dia

def matriz_de_dias(
    dados_estacao: pd.DataFrame,
) -> tuple[np.ndarray, pd.DatetimeIndex, dict[pd.Timestamp, pd.DataFrame]]:
    """Converte cada dia valido em 288 atributos e preserva suas grades."""
    linhas: list[np.ndarray] = []
    datas: list[pd.Timestamp] = []
    grades: dict[pd.Timestamp, pd.DataFrame] = {}

    chave_dia = dados_estacao["timestamp"].dt.normalize()
    for data, dados_dia in dados_estacao.groupby(chave_dia, sort=True):
        dia = grade_normalizada_do_dia(dados_dia)
        if dia is None:
            continue
        data = pd.Timestamp(data)
        linhas.append(
            np.r_[
                dia["irradiance_pu"].to_numpy(),
                dia["ambient_temperature_c"].to_numpy(),
            ]
        )
        datas.append(data)
        grades[data] = dia

    return np.asarray(linhas), pd.DatetimeIndex(datas), grades

## 5. Agrupamento sazonal

As variáveis são padronizadas por atributo e horário. Cada estação recebe dois clusters. O representante é um dia real, escolhido pela menor distância ao centroide.

In [ ]:
def selecionar_estacao(
    dados_estacao: pd.DataFrame, estacao: str
) -> tuple[pd.DataFrame, dict[pd.Timestamp, pd.DataFrame]]:
    """Seleciona dois medoides aproximados: observacoes mais proximas dos centroides."""
    X, datas, grades = matriz_de_dias(dados_estacao)
    if len(datas) < N_CLUSTERS_ESTACAO:
        raise ValueError(
            f"{estacao}: apenas {len(datas)} dias validos; "
            f"sao necessarios pelo menos {N_CLUSTERS_ESTACAO}"
        )

    meses_validos = set(ESTACOES[estacao])
    meses_encontrados = set(int(m) for m in datas.month)
    if not meses_encontrados.issubset(meses_validos):
        raise AssertionError(
            f"{estacao}: meses indevidos antes do K-means: {meses_encontrados}"
        )

    # Padronizacao por atributo/horario evita que a escala em graus Celsius
    # domine numericamente a irradiancia em pu.
    Xz = StandardScaler().fit_transform(X)
    modelo = KMeans(
        n_clusters=N_CLUSTERS_ESTACAO,
        random_state=SEED,
        n_init=N_INIT,
    )
    rotulos = modelo.fit_predict(Xz)

    escolhidos: list[dict[str, object]] = []
    for cluster in range(N_CLUSTERS_ESTACAO):
        indices = np.flatnonzero(rotulos == cluster)
        distancias = np.linalg.norm(
            Xz[indices] - modelo.cluster_centers_[cluster], axis=1
        )
        indice = int(indices[np.argmin(distancias)])
        data = pd.Timestamp(datas[indice])
        escolhidos.append(
            {
                "estacao": estacao,
                "cluster_kmeans": cluster,
                "data_representativa": data.date().isoformat(),
                "data_representativa_br": data.strftime("%d/%m/%Y"),
                "distancia_ao_centroide": float(distancias.min()),
                "numero_de_dias_no_cluster": int(len(indices)),
            }
        )

    resultado = pd.DataFrame(escolhidos)
    total = int(resultado["numero_de_dias_no_cluster"].sum())
    if total != len(datas):
        raise AssertionError(
            f"{estacao}: frequencias cobrem {total} de {len(datas)} dias validos"
        )

    # Regime 1 e o mais frequente. O desempate pela data torna a rotulagem
    # independente dos rotulos arbitrarios internos do K-means.
    resultado = resultado.sort_values(
        ["numero_de_dias_no_cluster", "data_representativa"],
        ascending=[False, True],
    ).reset_index(drop=True)
    resultado.insert(1, "regime", np.arange(1, len(resultado) + 1))
    resultado["frequencia_percentual"] = (
        100.0 * resultado["numero_de_dias_no_cluster"] / total
    ).round(2)
    resultado["peso"] = resultado["numero_de_dias_no_cluster"] / total

    for data_txt in resultado["data_representativa"]:
        data = pd.Timestamp(data_txt)
        if data.year != ANO_METEOROLOGICO or data.month not in meses_validos:
            raise AssertionError(
                f"Data sazonal inconsistente: {estacao}/{data.strftime('%d/%m/%Y')}"
            )
    return resultado, grades

## 6. Construção do JSON e dos arquivos auxiliares

In [ ]:
def construir_dicionario(
    resultados: pd.DataFrame,
    grades_por_estacao: dict[str, dict[pd.Timestamp, pd.DataFrame]],
) -> dict[str, dict[str, list[dict[str, float | str]]]]:
    """Cria o JSON diretamente da saida do K-means, sem datas manuais."""
    dicionario: dict[str, dict[str, list[dict[str, float | str]]]] = {}

    for estacao, meses in ESTACOES.items():
        dicionario[estacao] = {}
        linhas_estacao = resultados[resultados["estacao"].eq(estacao)]
        if len(linhas_estacao) != N_CLUSTERS_ESTACAO:
            raise AssertionError(
                f"{estacao}: esperados {N_CLUSTERS_ESTACAO} dias; "
                f"recebidos {len(linhas_estacao)}"
            )

        for linha in linhas_estacao.itertuples(index=False):
            data = pd.Timestamp(linha.data_representativa)
            if data.month not in meses:
                raise AssertionError(
                    f"{data.strftime('%d/%m/%Y')} nao pertence a {estacao}"
                )
            dia = grades_por_estacao[estacao][data]
            chave = data.strftime("%d/%m/%Y")
            registros: list[dict[str, float | str]] = []
            for timestamp, valores in dia.iterrows():
                registros.append(
                    {
                        "hora": timestamp.strftime("%H:%M"),
                        "irradiance_pu": float(valores["irradiance_pu"]),
                        "ambient_temperature_c": float(
                            valores["ambient_temperature_c"]
                        ),
                    }
                )
            if len(registros) != PONTOS_POR_DIA:
                raise AssertionError(f"{estacao}/{chave}: grade com tamanho invalido")
            dicionario[estacao][chave] = registros

    if sum(len(dias) for dias in dicionario.values()) != 8:
        raise AssertionError("O dicionario final deve conter exatamente oito dias")
    return dicionario

def dados_dos_dias_escolhidos(
    resultados: pd.DataFrame,
    grades_por_estacao: dict[str, dict[pd.Timestamp, pd.DataFrame]],
) -> pd.DataFrame:
    partes: list[pd.DataFrame] = []
    for linha in resultados.itertuples(index=False):
        data = pd.Timestamp(linha.data_representativa)
        parte = grades_por_estacao[linha.estacao][data].reset_index()
        parte.insert(0, "estacao", linha.estacao)
        parte.insert(1, "regime", int(linha.regime))
        parte.insert(2, "data_representativa", linha.data_representativa)
        partes.append(parte)
    return pd.concat(partes, ignore_index=True)

def gerar_grafico_estacao(
    resultado: pd.DataFrame,
    grades: dict[pd.Timestamp, pd.DataFrame],
    estacao: str,
    pasta_saida: Path,
) -> None:
    fig, eixos = plt.subplots(2, 1, figsize=(9.0, 6.2), sharex=True)
    cores = plt.cm.Set1.colors

    for i, linha in resultado.reset_index(drop=True).iterrows():
        data = pd.Timestamp(linha["data_representativa"])
        dia = grades[data]
        horas = np.arange(PONTOS_POR_DIA) / 6.0
        legenda = (
            f"Regime {int(linha['regime'])}: "
            f"{linha['data_representativa_br']} "
            f"({linha['frequencia_percentual']:.1f}%)"
        )
        eixos[0].plot(horas, dia["irradiance_pu"], color=cores[i], label=legenda)
        eixos[1].plot(
            horas,
            dia["ambient_temperature_c"],
            color=cores[i],
            label=legenda,
        )

    eixos[0].set_ylabel("Irradiancia [pu]")
    eixos[1].set_ylabel("Temperatura ambiente [degC]")
    eixos[1].set_xlabel("Hora do dia")
    eixos[1].set_xlim(0, 24)
    eixos[0].legend(fontsize=8)
    fig.suptitle(f"Dias meteorologicos representativos - {estacao}")
    fig.tight_layout()
    fig.savefig(
        pasta_saida / f"dias_representativos_{nome_simples(estacao)}.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.close(fig)

def gerar_grafico_frequencias(resultados: pd.DataFrame, pasta_saida: Path) -> None:
    fig, eixos = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
    for eixo, (estacao, _) in zip(eixos.flat, ESTACOES.items()):
        tabela = resultados[resultados["estacao"].eq(estacao)].sort_values("regime")
        rotulos = [
            f"Regime {int(l.regime)}\n{l.data_representativa_br}"
            for l in tabela.itertuples(index=False)
        ]
        valores = tabela["frequencia_percentual"].to_numpy()
        barras = eixo.barh(rotulos, valores, color=["#2878B5", "#F28E2B"])
        eixo.invert_yaxis()
        eixo.set_title(f"{estacao} (dias validos: {int(tabela.numero_de_dias_no_cluster.sum())})")
        eixo.grid(axis="x", linestyle=":", alpha=0.45)
        for barra, valor in zip(barras, valores):
            eixo.text(
                min(valor - 1.0, 95.0),
                barra.get_y() + barra.get_height() / 2,
                f"{valor:.1f}%",
                ha="right",
                va="center",
                color="white" if valor >= 20 else "black",
                fontweight="bold",
            )
    for eixo in eixos[-1, :]:
        eixo.set_xlabel("Frequencia entre os dias validos da estacao [%]")
    for eixo in eixos.flat:
        eixo.set_xlim(0, 100)
    fig.suptitle("Frequencia dos regimes meteorologicos sazonais")
    fig.tight_layout()
    fig.savefig(
        pasta_saida / "frequencia_regimes_sazonais.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.close(fig)

def validar_dicionario(
    dicionario: dict[str, dict[str, list[dict[str, float | str]]]]
) -> None:
    """Falha explicitamente diante de troca dia/mes ou grade incompleta."""
    horas_esperadas = [f"{10 * k // 60:02d}:{10 * k % 60:02d}" for k in range(144)]
    for estacao, meses in ESTACOES.items():
        for data_br, amostras in dicionario[estacao].items():
            data = pd.to_datetime(data_br, format="%d/%m/%Y", errors="raise")
            if data.year != ANO_METEOROLOGICO:
                raise AssertionError(f"Ano incorreto em {estacao}/{data_br}")
            if data.month not in meses:
                raise AssertionError(f"Mes incorreto em {estacao}/{data_br}")
            horas = [str(amostra["hora"]) for amostra in amostras]
            if horas != horas_esperadas:
                raise AssertionError(f"Grade horaria incorreta em {estacao}/{data_br}")

def salvar_resultados(
    resultados: pd.DataFrame,
    dados_escolhidos: pd.DataFrame,
    dicionario: dict[str, dict[str, list[dict[str, float | str]]]],
    pasta_saida: Path = PASTA_SAIDA,
) -> None:
    pasta_saida.mkdir(parents=True, exist_ok=True)
    resultados.to_csv(pasta_saida / "resumo_2_dias_por_estacao.csv", index=False)
    dados_escolhidos.to_csv(
        pasta_saida / "dados_dos_8_dias_representativos_sazonais.csv", index=False
    )
    with (pasta_saida / "dicionario_8_dias_representativos.json").open(
        "w", encoding="utf-8"
    ) as arquivo:
        json.dump(dicionario, arquivo, ensure_ascii=False, indent=2, allow_nan=False)

    manifesto = {
        "metodo": "K-means por estação; dia observado mais próximo do centroide",
        "ano_dos_dados": ANO_DADOS,
        "variaveis_do_agrupamento": [
            "irradiance_pu",
            "ambient_temperature_c",
        ],
        "observacao_variaveis": (
            "panel_temperature_c e generation_pu não entram no K-means porque "
            "são grandezas derivadas das variáveis meteorológicas usadas"
        ),
        "n_clusters_por_estacao": N_CLUSTERS_ESTACAO,
        "random_state": SEED,
        "n_init": N_INIT,
        "cobertura_minima_por_dia": COBERTURA_MINIMA,
        "estacoes": {nome: list(meses) for nome, meses in ESTACOES.items()},
        "dias_selecionados": {
            estacao: list(dias.keys()) for estacao, dias in dicionario.items()
        },
    }
    with (pasta_saida / "manifesto_dias_representativos.json").open(
        "w", encoding="utf-8"
    ) as arquivo:
        json.dump(manifesto, arquivo, ensure_ascii=False, indent=2)

## 7. Execução

A célula seguinte processa os 365 dias, gera oito representantes e grava os resultados no diretório configurado.

In [ ]:
def main() -> dict[str, dict[str, list[dict[str, float | str]]]]:
    PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
    dados = ler_arquivos(PASTA_CSV)

    resultados_estacoes: list[pd.DataFrame] = []
    grades_por_estacao: dict[str, dict[pd.Timestamp, pd.DataFrame]] = {}

    for estacao, meses in ESTACOES.items():
        dados_estacao = dados[dados["mes_arquivo"].isin(meses)].copy()
        resultado, grades = selecionar_estacao(dados_estacao, estacao)
        resultados_estacoes.append(resultado)
        grades_por_estacao[estacao] = grades
        gerar_grafico_estacao(resultado, grades, estacao, PASTA_SAIDA)
        print("\n" + resultado.to_string(index=False) + "\n")

    resultados = pd.concat(resultados_estacoes, ignore_index=True)
    dicionario = construir_dicionario(resultados, grades_por_estacao)
    validar_dicionario(dicionario)
    dados_escolhidos = dados_dos_dias_escolhidos(
        resultados, grades_por_estacao
    )
    gerar_grafico_frequencias(resultados, PASTA_SAIDA)
    salvar_resultados(resultados, dados_escolhidos, dicionario, PASTA_SAIDA)

    print(f"Arquivos salvos em: {PASTA_SAIDA}")
    print("Dias selecionados:")
    for estacao, dias in dicionario.items():
        print(f"  {estacao}: {', '.join(dias)}")
    return dicionario

dicionario_dias = main()

## Saídas principais

- `dicionario_8_dias_representativos.json`;
- `resumo_2_dias_por_estacao.csv`;
- `dados_dos_8_dias_representativos_sazonais.csv`;
- `manifesto_dias_representativos.json`;
- figuras dos representantes e de suas frequências.

O JSON gerado pode substituir `data/meteorology/dicionario_8_dias_representativos.json` após a conferência dos resultados.